In [2]:
import os
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

from datetime import timedelta
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
import plotly.graph_objects as go


import itertools

In [3]:
directory = '../datasets/year_month=22-09/plugin=slurm_pub/metric=s21.cluster_cpu_util/a_0.parquet'
cpu_util_df = pd.read_parquet(directory)
mem_util_dir = '../datasets/year_month=22-09/plugin=slurm_pub/metric=s21.cluster_mem_util/a_0.parquet'
mem_util_df = pd.read_parquet(mem_util_dir)
directory = '../datasets/year_month=22-09/plugin=slurm_pub/metric=s21.cluster_gpu_util/a_0.parquet'
gpu_util_df = pd.read_parquet(directory)
jobid_dir = '../datasets/year_month=22-09/plugin=slurm_pub/metric=job_id/a_0.parquet'
job_id_df = pd.read_parquet(jobid_dir)
jobtable_dir = '../datasets/year_month=22-09/plugin=job_table/metric=job_info_marconi100/a_0.parquet'
job_table_df = pd.read_parquet(jobtable_dir)

In [6]:
import plotly.graph_objects as go
import ruptures as rpt


def plot_res_util_pct_with_waste_plotly_pelt(
    user_id,
    job_id,
    util_df,
    resource,
    # ---- PELT params ----
    show_pelt=True,
    pelt_model="l2",      # "l1", "l2", "rbf"
    pelt_pen=10,          # higher => fewer change points
    pelt_min_size=3,
    pelt_jump=1,
    # ---- optional time padding ----
    pad_seconds=0,        # e.g., 30 to extend window by +/- 30s
    # ---- display ----
    show_fig=True
):
    # -------------------------
    # Filter job metadata
    # -------------------------
    user_df_1 = job_table_df[job_table_df["user_id"] == user_id]
    user_df_2 = job_id_df[job_id_df["user_id"] == user_id]

    start_time = pd.to_datetime(
        user_df_1[user_df_1["job_id"] == job_id]["start_time"].iloc[0],
        utc=True, errors="coerce"
    )
    end_time = pd.to_datetime(
        user_df_1[user_df_1["job_id"] == job_id]["end_time"].iloc[0],
        utc=True, errors="coerce"
    )

    if pd.isna(start_time) or pd.isna(end_time):
        print("Missing/invalid start_time or end_time for this job.")
        return None

    # ✅ safe padding
    if pad_seconds and pad_seconds > 0:
        pad = pd.Timedelta(seconds=int(pad_seconds))
        start_time = start_time - pad
        end_time = end_time + pad

    partition_num = user_df_2[user_df_2["value"] == job_id]["partition"].iloc[0]

    # -------------------------
    # Ensure util_df timestamp is datetime (tz-aware OK)
    # -------------------------
    if not pd.api.types.is_datetime64_any_dtype(util_df["timestamp"]):
        util_df = util_df.copy()
        util_df["timestamp"] = pd.to_datetime(util_df["timestamp"], utc=True, errors="coerce")

    # If timestamp is datetime but tz-naive, make it UTC to match start/end
    # (If already tz-aware, this is a no-op)
    if getattr(util_df["timestamp"].dtype, "tz", None) is None:
        util_df = util_df.copy()
        util_df["timestamp"] = util_df["timestamp"].dt.tz_localize("UTC")

    # -------------------------
    # Utilization slice
    # -------------------------
    x_df = util_df[
        (util_df["timestamp"] >= start_time)
        & (util_df["timestamp"] <= end_time)
        & (util_df["partition"] == partition_num)
    ].copy()

    if x_df.empty:
        print("No utilization data found.")
        return None

    x_df = x_df.dropna(subset=["timestamp", "value"]).sort_values("timestamp")
    x_df["value"] = x_df["value"].astype(float).clip(lower=0, upper=100)

    # -------------------------
    # Plotly figure
    # -------------------------
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=x_df["timestamp"],
        y=x_df["value"],
        mode="lines",
        name=f"{resource} utilized (%)",
        hovertemplate="Time: %{x}<br>Util: %{y:.2f}%<extra></extra>"
    ))

    fig.add_trace(go.Scatter(
        x=x_df["timestamp"],
        y=np.full(len(x_df), 100.0),
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=x_df["timestamp"],
        y=x_df["value"],
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        name="Idle capacity gap",
        customdata=(100.0 - x_df["value"]).to_numpy(),
        hovertemplate="Time: %{x}<br>Idle gap: %{customdata:.2f}%<extra></extra>"
    ))

    fig.add_hline(y=100, line_dash="dash")

    # -------------------------
    # ✅ PELT change points + overlay
    # -------------------------
    bkps = []
    if show_pelt:
        y = x_df["value"].to_numpy(dtype=float).reshape(-1, 1)

        if len(y) >= max(5, pelt_min_size * 2):
            algo = rpt.Pelt(model=pelt_model, min_size=pelt_min_size, jump=pelt_jump).fit(y)
            bkps = algo.predict(pen=pelt_pen)  # includes last point == len(y)

            bkps_plot = [b for b in bkps if b < len(x_df)]
            for b in bkps_plot:
                # ✅ convert Timestamp -> python datetime (prevents Plotly sum(Timestamp) bug)
                t = x_df["timestamp"].iloc[b].to_pydatetime()
                fig.add_vline(x=t, line_dash="dot", line_width=2)
        else:
            print("Not enough points for PELT on this job window.")

    fig.update_layout(
        title=f"{resource} Utilization (user={user_id}, job={job_id}, partition={partition_num})",
        xaxis=dict(title="Time", rangeslider=dict(visible=True), type="date"),
        yaxis=dict(title="Utilization (%)", range=[0, 105]),
        hovermode="x unified",
        legend=dict(orientation="h"),
        template="plotly_white",
        margin=dict(l=60, r=20, t=120, b=60)
    )

    if show_fig:
        fig.show()

    return fig, x_df, partition_num, start_time, end_time, bkps


def run_plot_interactive_plotly_pelt():
    user_id = int(input("Enter a user ID: "))
    job_id = int(input("Enter a job ID: "))
    resource = input("MEMORY / CPU / GPU: ").strip().upper()

    valid1 = ((job_table_df["user_id"] == user_id) & (job_table_df["job_id"] == job_id)).any()
    valid2 = ((job_id_df["user_id"] == user_id) & (job_id_df["value"] == job_id)).any()

    if not valid1 or not valid2:
        print("Invalid user/job combination.")
        return None

    if resource == "MEMORY":
        util_df = mem_util_df
    elif resource == "CPU":
        util_df = cpu_util_df
    elif resource == "GPU":
        util_df = gpu_util_df
    else:
        print("Invalid resource.")
        return None

    out = plot_res_util_pct_with_waste_plotly_pelt(
        user_id=user_id,
        job_id=job_id,
        util_df=util_df,
        resource=resource,
        show_pelt=True,
        pelt_model="rbf",
        pelt_pen=12,
        pelt_min_size=6,
        pelt_jump=1,
        pad_seconds=0,
        show_fig=True
    )

    if out is None:
        return None

    fig, x_df, part, start, end, bkps = out
    print(f"Plotted: user={user_id}, job={job_id}, resource={resource}, partition={part}, window={start} → {end}, bkps={bkps}")
    return out

In [13]:
run_plot_interactive_plotly_pelt()

Enter a user ID: 2
Enter a job ID: 1582167
MEMORY / CPU / GPU: MEMORY


Plotted: user=2, job=1582167, resource=MEMORY, partition=1, window=2022-09-15 06:11:39+00:00 → 2022-09-15 14:31:41+00:00, bkps=[221, 500, 947, 1261, 1713, 2429, 2553, 2952]


(Figure({
     'data': [{'hovertemplate': 'Time: %{x}<br>Util: %{y:.2f}%<extra></extra>',
               'mode': 'lines',
               'name': 'MEMORY utilized (%)',
               'type': 'scatter',
               'x': array([datetime.datetime(2022, 9, 15, 6, 11, 40, tzinfo=<UTC>),
                           datetime.datetime(2022, 9, 15, 6, 11, 50, tzinfo=<UTC>),
                           datetime.datetime(2022, 9, 15, 6, 12, tzinfo=<UTC>), ...,
                           datetime.datetime(2022, 9, 15, 14, 31, 20, tzinfo=<UTC>),
                           datetime.datetime(2022, 9, 15, 14, 31, 30, tzinfo=<UTC>),
                           datetime.datetime(2022, 9, 15, 14, 31, 40, tzinfo=<UTC>)], dtype=object),
               'y': array([53.22534277, 53.11449096, 53.11772717, ..., 74.21503063, 74.21503063,
                           74.10417882])},
              {'hoverinfo': 'skip',
               'line': {'width': 0},
               'mode': 'lines',
               'showlegend': 